# Chương 13: Học biểu diễn quy mô lớn và Mô hình nền tảng

Notebook này minh họa thực hành hai cơ chế kỹ thuật then chốt thúc đẩy các mô hình nền tảng hiện đại:
1. **Kỹ thuật thích ứng hạng thấp LoRA (Low-Rank Adaptation)**: Tự cài đặt tầng tuyến tính LoRA từ đầu bằng NumPy, tính toán đạo hàm giải tích để cập nhật ma trận hạng thấp $A$ và $B$, chứng minh giảm thiểu $98\%$ tham số cần huấn luyện và kiểm chứng kỹ thuật gộp trọng số (weight merging).
2. **Học biểu diễn tương phản (Contrastive Representation Learning) & Hàm mất mát InfoNCE**: Cài đặt hàm mất mát InfoNCE từ đầu, huấn luyện tầng chiếu đặc trưng và trực quan hóa ma trận tương đồng (similarity matrix) trước và sau khi học.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Thiết lập hạt giống ngẫu nhiên để tái lập kết quả
np.random.seed(42)
print("Môi trường sẵn sàng: NumPy", np.__version__)

## 1. Cài đặt LoRA (Low-Rank Adaptation) từ đầu

### Cơ sở toán học
Cho một tầng tuyến tính đã tiền huấn luyện với ma trận trọng số đóng băng $W_0 \in \mathbb{R}^{d_{out} \times d_{in}}$. Khi tinh chỉnh cho một nhiệm vụ cụ thể, biến thiên trọng số $\Delta W$ được phân rã thành:
$$\Delta W = \frac{\alpha}{r} B \cdot A$$
Trong đó:
- $A \in \mathbb{R}^{r \times d_{in}}$ được khởi tạo theo phân phối Gauss $\mathcal{N}(0, \sigma^2)$.
- $B \in \mathbb{R}^{d_{out} \times r}$ được khởi tạo bằng $0$.
- $r \ll \min(d_{in}, d_{out})$ là thứ hạng thích ứng (rank).
- $\alpha$ là hằng số co dãn (scaling hyperparameter).

Đầu ra của tầng đối với đầu vào $X \in \mathbb{R}^{N \times d_{in}}$ là:
$$H = X W_0^T + \frac{\alpha}{r} X A^T B^T$$

In [ ]:
class LoRALinearLayer:
    def __init__(self, in_features, out_features, rank=4, alpha=8.0):
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        
        # Trọng số tiền huấn luyện W0 (bị đóng băng hoàn toàn - không cập nhật gradient)
        self.W0 = np.random.randn(out_features, in_features) * 0.05
        
        # Hai ma trận hạng thấp của LoRA
        # A: khởi tạo Gaussian
        self.A = np.random.randn(rank, in_features) * (1.0 / np.sqrt(in_features))
        # B: khởi tạo bằng 0 để tại bước 0, Delta W = 0 hoàn hảo
        self.B = np.zeros((out_features, rank))
        
        # Lưu trữ gradient
        self.grad_A = None
        self.grad_B = None
        
        # Bộ nhớ đệm cho lan truyền ngược
        self.cache_X = None
        
    def forward(self, X):
        """
        Lan truyền tiến:
        H = X @ W0.T + scaling * (X @ A.T @ B.T)
        """
        self.cache_X = X
        # Đầu ra từ mô hình cơ sở đóng băng
        base_out = X @ self.W0.T
        
        # Đầu ra từ nhánh LoRA hạng thấp
        lora_out = (X @ self.A.T) @ self.B.T * self.scaling
        
        return base_out + lora_out
    
    def backward(self, grad_output):
        """
        Lan truyền ngược tính gradient cho A và B:
        grad_output có kích thước (N, out_features)
        """
        N = self.cache_X.shape[0]
        X = self.cache_X  # (N, in_features)
        
        # Nhánh trung gian: Z_A = X @ A.T (N, rank)
        Z_A = X @ self.A.T
        
        # Gradient theo ma trận B:
        # dL/dB = scaling * (grad_output.T @ Z_A)
        self.grad_B = self.scaling * (grad_output.T @ Z_A)
        
        # Gradient theo ma trận A:
        # dL/dZ_A = grad_output @ B * scaling
        # dL/dA = (dL/dZ_A).T @ X
        grad_Z_A = (grad_output @ self.B) * self.scaling
        self.grad_A = grad_Z_A.T @ X
        
    def step(self, lr=1e-2):
        """Cập nhật trọng số của riêng các ma trận LoRA"""
        self.A -= lr * self.grad_A
        self.B -= lr * self.grad_B
        
    def merge_weights(self):
        """
        Gộp nhánh LoRA trực tiếp vào ma trận W0 phục vụ suy luận không độ trễ
        W_merged = W0 + scaling * (B @ A)
        """
        delta_W = self.scaling * (self.B @ self.A)
        return self.W0 + delta_W

# Khảo sát số lượng tham số tiết kiệm được
in_dim, out_dim, rank = 512, 512, 4
layer = LoRALinearLayer(in_dim, out_dim, rank=rank, alpha=8.0)

w0_params = in_dim * out_dim
lora_params = (rank * in_dim) + (out_dim * rank)
reduction = (1 - lora_params / w0_params) * 100

print(f"Số tham số mô hình gốc W0: {w0_params:,}")
print(f"Số tham số huấn luyện LoRA (A + B, rank={rank}): {lora_params:,}")
print(f"Tỷ lệ giảm thiểu tham số cần huấn luyện: {reduction:.2f}%")

### Huấn luyện LoRA trên một tác vụ thích ứng (Task Adaptation)
Giả sử mô hình tiền huấn luyện $W_0$ nhận đầu vào $X$ và ta muốn điều chỉnh nó để sinh ra nhãn mục tiêu $Y_{target}$ tương ứng với một phân phối dữ liệu mới.

In [ ]:
# Tạo dữ liệu nhiệm vụ mới với một phép biến đổi mục tiêu
N_samples = 200
X_train = np.random.randn(N_samples, in_dim)
# Mục tiêu là phép biến đổi hơi khác so với W0
W_target = layer.W0 + np.random.randn(out_dim, in_dim) * 0.02
Y_train = X_train @ W_target.T

# Vòng lặp huấn luyện LoRA
losses = []
epochs = 80
lr = 0.05

for epoch in range(epochs):
    # Lan truyền tiến
    preds = layer.forward(X_train)
    # Mean Squared Error
    loss = np.mean((preds - Y_train) ** 2)
    losses.append(loss)
    
    # Đạo hàm MSE: dL/dPred = 2 * (preds - Y) / (N * out_dim)
    grad_output = 2.0 * (preds - Y_train) / (N_samples * out_dim)
    
    # Lan truyền ngược và cập nhật riêng A, B
    layer.backward(grad_output)
    layer.step(lr=lr)

# Kiểm chứng việc gộp trọng số (Weight Merging)
W_merged = layer.merge_weights()
preds_merged = X_train @ W_merged.T
diff_merged = np.max(np.abs(preds_merged - layer.forward(X_train)))

print(f"Mất mát ban đầu: {losses[0]:.6f} -> Mất mát sau {epochs} epochs: {losses[-1]:.6f}")
print(f"Sai lệch tối đa giữa suy luận LoRA và ma trận gộp W_merged: {diff_merged:.10e}")

plt.figure(figsize=(7, 4))
plt.plot(losses, color="royalblue", lw=2, label="LoRA Training Loss (MSE)")
plt.title("Đường cong hội tụ khi tinh chỉnh với LoRA (Chỉ cập nhật 1.56% tham số)")
plt.xlabel("Kỷ nguyên (Epochs)")
plt.ylabel("Loss")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

## 2. Học biểu diễn tương phản (Contrastive Learning) & Hàm mất mát InfoNCE

### Cơ sở toán học của InfoNCE
Trong phương pháp học tương phản (như SimCLR, CLIP), hai biến thể tăng cường (augmented views) của cùng một mẫu được coi là một **cặp dương** $(z_i, z_j)$.
Hàm mất mát InfoNCE với nhiệt độ $\tau$:
$$\mathcal{L}_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j) / \tau)}{\sum_{k=1}^{2N} \exp(\text{sim}(z_i, z_k) / \tau)}$$
Trong đó $\text{sim}(u, v) = \frac{u^T v}{\|u\|_2 \|v\|_2}$ là Cosine Similarity.

In [ ]:
def compute_infonce_loss(features_view1, features_view2, temperature=0.1):
    """
    features_view1: (N, D) biểu diễn của góc nhìn thứ nhất
    features_view2: (N, D) biểu diễn của góc nhìn thứ hai (cặp dương tương ứng theo chỉ số hàng)
    """
    N = features_view1.shape[0]
    
    # 1. Chuẩn hóa L2 về vector đơn vị
    z1 = features_view1 / np.linalg.norm(features_view1, axis=1, keepdims=True)
    z2 = features_view2 / np.linalg.norm(features_view2, axis=1, keepdims=True)
    
    # Gộp toàn bộ biểu diễn thành kích thước (2N, D)
    z = np.vstack([z1, z2])
    
    # 2. Ma trận tương đồng Cosine giữa mọi cặp: (2N, 2N)
    sim_matrix = (z @ z.T) / temperature
    
    # Trừ đi giá trị lớn nhất trên mỗi hàng để tránh tràn số học (numerical stability)
    sim_matrix -= np.max(sim_matrix, axis=1, keepdims=True)
    
    # Mặt nạ loại bỏ chính phần tử đối xứng trên đường chéo chính (tự so sánh với chính nó)
    mask = np.eye(2 * N, dtype=bool)
    sim_matrix[mask] = -1e9
    
    # Chỉ số cặp dương: vị trí i có cặp dương là i + N (và ngược lại)
    pos_indices = np.concatenate([np.arange(N, 2 * N), np.arange(0, N)])
    
    # Tính Softmax Cross-Entropy đối ngẫu
    exp_sim = np.exp(sim_matrix)
    denominator = np.sum(exp_sim, axis=1)
    numerator = exp_sim[np.arange(2 * N), pos_indices]
    
    loss = -np.mean(np.log(numerator / (denominator + 1e-12)))
    return loss, z @ z.T

# Mô phỏng dữ liệu 10 đối tượng mẫu qua 2 góc nhìn bị nhiễu
N_items = 10
latent_dim = 16
base_concepts = np.random.randn(N_items, latent_dim)

# Tạo 2 views với nhiễu ngẫu nhiên
view_A = base_concepts + np.random.randn(N_items, latent_dim) * 0.3
view_B = base_concepts + np.random.randn(N_items, latent_dim) * 0.3

# Tính ma trận tương đồng ban đầu
loss_init, sim_matrix_raw = compute_infonce_loss(view_A, view_B, temperature=0.2)
print(f"InfoNCE Loss ban đầu (trước tinh chỉnh): {loss_init:.4f}")

### Trực quan hóa ma trận tương đồng Cosine Similarity
Trong mô hình biểu diễn chất lượng cao, các ô trên đường chéo phụ tương ứng cặp dương $(i, i+N)$ và $(i+N, i)$ phải có giá trị tiệm cận $1.0$, trong khi các cặp âm ngẫu nhiên khác tiệm cận $0$ hoặc âm.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim_matrix_raw, cmap="viridis")
plt.colorbar(im, ax=ax, label="Cosine Similarity")
ax.set_title(f"Ma trận tương đồng giữa các góc nhìn (2N={2*N_items})")
ax.set_xlabel("Chỉ số biểu diễn (0..9: View A, 10..19: View B)")
ax.set_ylabel("Chỉ số biểu diễn")
plt.tight_layout()
plt.show()

print("Hoàn tất thực hành Chapter 13: Đã kiểm chứng cơ chế LoRA và InfoNCE Contrastive Loss.")